# 🎬 Recap Studio v3 AUTO — free hosting on Google Colab

This notebook runs your **Recap Studio v3 AUTO** website for free — no credit card.
Upload a video on the page → it auto-transcribes, auto-translates, dubs it with an
AI voice, burns subtitles and renders a finished MP4 to download.

**How to use:**
1. In Colab: **Runtime → Run all** (run each code cell top to bottom).
2. Wait for the last cell to print **YOUR WEBSITE IS LIVE AT:** + a link.
3. Open that link on your phone → upload a clip → ✨ Generate → Download.
4. Keep this Colab tab open — closing it stops the website.

**Limits (honest):** each run gives a *new* link; a session lasts up to ~12 hours;
the first run installs packages and downloads the Whisper model (~0.5 GB).
Short clips (under ~3 minutes) work best — free Colab is CPU-only and slow.

In [ ]:
# Step 1 — install what the website needs (ffmpeg is already on Colab)
# av is pinned: faster-whisper is incompatible with PyAV >= 15
%pip install -q fastapi "uvicorn[standard]" python-multipart faster-whisper "av==14.2.0" edge-tts gtts deep-translator
import shutil; assert shutil.which('ffmpeg'), 'ffmpeg missing'; print('OK')


In [ ]:
%%writefile pipeline.py
"""Recap Studio v3 AUTO — the automatic dubbing pipeline.

Steps (all local, driven by ffmpeg):
  1. Extract the video's audio (16 kHz mono wav)
  2. Transcribe it with faster-whisper -> timed segments
  3. Translate each segment to the target language (deep-translator)
  4. Speak each translated segment with edge-tts, falling back to gTTS
     (per-segment, failure-proof; provider used is logged to tts.log)
  5. Fit every voice clip into its original time window and assemble one
     full-length dub track that lines up with the picture
  6. Burn the translated subtitles into the video and render the final MP4

The pipeline never raises for a single bad segment: a failed translation
keeps the original text, a failed TTS segment becomes silence. Only truly
fatal problems (no audio, unreadable file, ffmpeg missing) fail the job.
"""
import asyncio
import json
import os
import re
import subprocess

# Sanitize proxy env vars (some HTTP libs crash on bracketed IPv6 entries)
for _var in ("no_proxy", "NO_PROXY"):
    if os.environ.get(_var):
        _clean = [e.strip() for e in os.environ[_var].split(",") if "[" not in e and "]" not in e]
        os.environ[_var] = ",".join(_clean)

# If the runtime provides its own CA bundle (e.g. behind a TLS-intercepting
# egress proxy, as in some sandboxes), point requests-based libraries at it.
# Harmless on Colab / normal networks, where SSL_CERT_FILE is usually unset.
_ca_bundle = os.environ.get("SSL_CERT_FILE")
if _ca_bundle and os.path.exists(_ca_bundle):
    os.environ.setdefault("REQUESTS_CA_BUNDLE", _ca_bundle)

WHISPER_MODEL = os.environ.get("WHISPER_MODEL", "small")

# ---------------------------------------------------------------- languages
# ONE central map for every language we support. For each language:
#   google   – code for deep-translator's Google Translate
#   mymemory – locale code for the MyMemory fallback translator
#   whisper  – code faster-whisper accepts for transcription (source side)
#   gtts     – gTTS language code, or None where gTTS has no such voice
#              (that language is then edge-tts only, silence after that)
#   edge     – Microsoft edge-tts neural voices (female + male)
# Codes below were verified against the installed libraries: Google and
# gTTS both call Filipino "tl" (Tagalog); gTTS has NO Lao, so lo is
# edge-only. Tedim/Zomi (ctd) is deliberately NOT here — no translator or
# TTS supports it, so offering it as an auto target would be fake.
def _L(google, mymemory, whisper, gtts, female, male):
    return {"google": google, "mymemory": mymemory, "whisper": whisper,
            "gtts": gtts, "edge": {"female": female, "male": male}}


LANGS = {
    "my":    _L("my", "my-MM", "my", "my", "my-MM-NilarNeural", "my-MM-ThihaNeural"),
    "en":    _L("en", "en-GB", "en", "en", "en-US-AriaNeural", "en-US-GuyNeural"),
    "th":    _L("th", "th-TH", "th", "th", "th-TH-PremwadeeNeural", "th-TH-NiwatNeural"),
    "zh-CN": _L("zh-CN", "zh-CN", "zh", "zh-CN", "zh-CN-XiaoxiaoNeural", "zh-CN-YunyangNeural"),
    "ja":    _L("ja", "ja-JP", "ja", "ja", "ja-JP-NanamiNeural", "ja-JP-KeitaNeural"),
    "ko":    _L("ko", "ko-KR", "ko", "ko", "ko-KR-SunHiNeural", "ko-KR-InJoonNeural"),
    "hi":    _L("hi", "hi-IN", "hi", "hi", "hi-IN-SwaraNeural", "hi-IN-MadhurNeural"),
    "id":    _L("id", "id-ID", "id", "id", "id-ID-GadisNeural", "id-ID-ArdiNeural"),
    "vi":    _L("vi", "vi-VN", "vi", "vi", "vi-VN-HoaiMyNeural", "vi-VN-NamMinhNeural"),
    "ms":    _L("ms", "ms-MY", "ms", "ms", "ms-MY-YasminNeural", "ms-MY-OsmanNeural"),
    "es":    _L("es", "es-ES", "es", "es", "es-ES-ElviraNeural", "es-ES-AlvaroNeural"),
    "fr":    _L("fr", "fr-FR", "fr", "fr", "fr-FR-DeniseNeural", "fr-FR-HenriNeural"),
    "de":    _L("de", "de-DE", "de", "de", "de-DE-KatjaNeural", "de-DE-ConradNeural"),
    "ar":    _L("ar", "ar-SA", "ar", "ar", "ar-SA-ZariyahNeural", "ar-SA-HamedNeural"),
    "bn":    _L("bn", "bn-IN", "bn", "bn", "bn-IN-TanishaaNeural", "bn-IN-BashkarNeural"),
    "ta":    _L("ta", "ta-IN", "ta", "ta", "ta-IN-PallaviNeural", "ta-IN-ValluvarNeural"),
    "te":    _L("te", "te-IN", "te", "te", "te-IN-ShrutiNeural", "te-IN-MohanNeural"),
    "ur":    _L("ur", "ur-PK", "ur", "ur", "ur-PK-UzmaNeural", "ur-PK-AsadNeural"),
    "pt":    _L("pt", "pt-BR", "pt", "pt", "pt-BR-FranciscaNeural", "pt-BR-AntonioNeural"),
    "ru":    _L("ru", "ru-RU", "ru", "ru", "ru-RU-SvetlanaNeural", "ru-RU-DmitryNeural"),
    "it":    _L("it", "it-IT", "it", "it", "it-IT-ElsaNeural", "it-IT-DiegoNeural"),
    "fil":   _L("tl", "fil-PH", "tl", "tl", "fil-PH-BlessicaNeural", "fil-PH-AngeloNeural"),
    "km":    _L("km", "km-KH", "km", "km", "km-KH-SreymomNeural", "km-KH-PisethNeural"),
    "lo":    _L("lo", "lo-LA", "lo", None, "lo-LA-KeomanyNeural", "lo-LA-ChanthavongNeural"),
}
DEFAULT_VOICE = {"female": "en-US-AriaNeural", "male": "en-US-GuyNeural"}

# Normalise whatever the UI / user sends ("my", "Burmese", "burmese", "en"...)
# to one of the LANGS keys above.
LANG_ALIASES = {
    "burmese": "my", "myanmar": "my", "my": "my",
    "english": "en", "en": "en",
    "thai": "th", "th": "th",
    "hindi": "hi", "hi": "hi",
    "korean": "ko", "ko": "ko",
    "japanese": "ja", "ja": "ja",
    "chinese": "zh-CN", "mandarin": "zh-CN", "zh": "zh-CN", "zh-cn": "zh-CN",
    "indonesian": "id", "id": "id",
    "vietnamese": "vi", "vi": "vi",
    "malay": "ms", "ms": "ms",
    "spanish": "es", "es": "es",
    "french": "fr", "fr": "fr",
    "german": "de", "de": "de",
    "arabic": "ar", "ar": "ar",
    "bengali": "bn", "bn": "bn",
    "tamil": "ta", "ta": "ta",
    "telugu": "te", "te": "te",
    "urdu": "ur", "ur": "ur",
    "portuguese": "pt", "pt": "pt",
    "russian": "ru", "ru": "ru",
    "italian": "it", "it": "it",
    "filipino": "fil", "tagalog": "fil", "fil": "fil", "tl": "fil",
    "khmer": "km", "km": "km",
    "lao": "lo", "lo": "lo",
    "tedim": "my", "zomi": "my",  # no Tedim ASR/TTS exists; Burmese is the honest fallback
}


def norm_lang(code):
    if not code:
        return "auto"
    return LANG_ALIASES.get(str(code).strip().lower(), str(code).strip().lower())


def pick_voice(lang, gender):
    entry = LANGS.get(lang)
    if entry:
        return entry["edge"].get(gender, DEFAULT_VOICE["female"])
    return DEFAULT_VOICE.get(gender, DEFAULT_VOICE["female"])


def whisper_code(lang):
    """faster-whisper code for a normalised language, or None for auto."""
    if lang in ("auto", "", None):
        return None
    entry = LANGS.get(lang)
    if entry:
        return entry["whisper"]
    code = str(lang).split("-")[0]
    return code or None


# ---------------------------------------------------------------- ffmpeg helpers
def run(cmd, **kw):
    """Run a command, raising a readable error on failure."""
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, **kw)
    if r.returncode != 0:
        tail = (r.stderr or "").strip().splitlines()[-3:]
        raise RuntimeError("Command failed: " + " ".join(cmd[:2]) + " … " + " | ".join(tail))
    return r


def probe_duration(path):
    r = run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
             "-of", "json", path])
    return float(json.loads(r.stdout)["format"]["duration"])


def probe_has_audio(path):
    r = run(["ffprobe", "-v", "error", "-select_streams", "a",
             "-show_entries", "stream=index", "-of", "csv=p=0", path])
    return bool(r.stdout.strip())


def srt_time(sec):
    sec = max(0.0, float(sec))
    h = int(sec // 3600); m = int((sec % 3600) // 60)
    s = int(sec % 60); ms = int(round((sec - int(sec)) * 1000))
    if ms == 1000:
        s += 1; ms = 0
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def write_srt(segments, path):
    with open(path, "w", encoding="utf-8") as f:
        for i, seg in enumerate(segments, 1):
            f.write(f"{i}\n{srt_time(seg['start'])} --> {srt_time(seg['end'])}\n"
                    f"{seg['dub_text']}\n\n")


# ---------------------------------------------------------------- step 2: ASR
_whisper_model = None

def transcribe(audio_wav, source_lang):
    """Return (segments, detected_lang). Segments: {start, end, text}."""
    global _whisper_model
    from faster_whisper import WhisperModel
    if _whisper_model is None:
        _whisper_model = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")
    lang = whisper_code(source_lang)  # zh-CN -> zh, fil -> tl, auto -> None
    segs, info = _whisper_model.transcribe(audio_wav, language=lang, vad_filter=True)
    out = []
    for s in segs:
        text = (s.text or "").strip()
        if text:
            out.append({"start": float(s.start), "end": float(s.end), "text": text})
    return out, (info.language or source_lang or "en")


# ---------------------------------------------------------------- step 3: translate
def translate_segments(segments, source_lang, target_lang, log):
    """Fill seg['dub_text'] with the translation (or original if skipped/failed)."""
    if source_lang == target_lang:
        for seg in segments:
            seg["dub_text"] = seg["text"]
        return
    from deep_translator import GoogleTranslator, MyMemoryTranslator
    # Codes come from the central LANGS map (e.g. Filipino is "tl" for
    # Google, "fil-PH" for MyMemory). Unknown codes pass through as-is.
    def gcode(c):
        return LANGS.get(c, {}).get("google", c)

    def mmcode(c):
        return LANGS.get(c, {}).get("mymemory", c)

    src = "auto" if source_lang in ("auto", "", None) else source_lang
    mm_src = "en-GB" if src == "auto" else mmcode(src)
    mm_tgt = mmcode(target_lang)
    google = GoogleTranslator(source=gcode(src) if src != "auto" else "auto",
                              target=gcode(target_lang))
    for seg in segments:
        text = seg["text"]
        try:
            seg["dub_text"] = google.translate(text) or text
            continue
        except Exception as e:
            log(f"Google translate failed for one line ({e}); trying MyMemory")
        try:
            seg["dub_text"] = MyMemoryTranslator(source=mm_src, target=mm_tgt).translate(text) or text
        except Exception as e:
            log(f"Translation failed for one line ({e}); keeping original text")
            seg["dub_text"] = text


# ---------------------------------------------------------------- step 4: TTS
# Fallback chain per segment: edge-tts first (best voices), then gTTS
# (Google Translate TTS — simpler, one voice per language, no gender
# choice, but reachable from networks where Microsoft's edge-tts WSS
# endpoint is blocked with a 403). A segment only becomes silence if
# every provider fails.
MIN_TTS_BYTES = 1024  # a real spoken line is never a few hundred bytes

# Languages gTTS can speak, derived from the central LANGS map
# (languages whose map entry has gtts=None — currently only Lao — are
# edge-tts only and fall back to silence if edge is unreachable).
GTTS_LANGS = {code: e["gtts"] for code, e in LANGS.items() if e["gtts"]}


def _mp3_ok(path):
    return os.path.exists(path) and os.path.getsize(path) >= MIN_TTS_BYTES


async def _tts_async(text, voice, out_path):
    import ssl
    import edge_tts
    # edge-tts pins its SSL context to the certifi bundle only, which fails
    # behind TLS-intercepting proxies (self-signed chain). Swap in the system
    # default context, which also honours SSL_CERT_FILE. Safe everywhere.
    try:
        edge_tts.communicate._SSL_CTX = ssl.create_default_context()
    except Exception:
        pass
    proxy = os.environ.get("https_proxy") or os.environ.get("HTTPS_PROXY") or None
    await edge_tts.Communicate(text, voice, proxy=proxy).save(out_path)


def tts_segment(text, voice, out_path):
    asyncio.run(_tts_async(text, voice, out_path))


def _tts_gtts(text, target_lang, out_path):
    from gtts import gTTS
    code = GTTS_LANGS.get(target_lang)
    if not code:
        raise ValueError(f"gTTS has no voice for language '{target_lang}'")
    gTTS(text=text, lang=code).save(out_path)


def tts_segment_fallback(text, voice, target_lang, out_path, log):
    """Voice one segment. Returns 'edge-tts', 'gtts' or None (caller uses
    silence). Never raises."""
    try:
        if os.path.exists(out_path):
            os.remove(out_path)
        tts_segment(text, voice, out_path)
        if _mp3_ok(out_path):
            return "edge-tts"
        log(f"edge-tts produced an empty/tiny file ({os.path.getsize(out_path) if os.path.exists(out_path) else 0} bytes); trying gTTS")
    except Exception as e:
        log(f"edge-tts failed ({e}); trying gTTS")
    try:
        if os.path.exists(out_path):
            os.remove(out_path)
        _tts_gtts(text, target_lang, out_path)
        if _mp3_ok(out_path):
            return "gtts"
        log(f"gTTS produced an empty/tiny file; line will be silent")
    except Exception as e:
        log(f"gTTS failed ({e}); line will be silent")
    return None


def atempo_chain(speed):
    """ffmpeg atempo only takes 0.5–2.0 per instance; chain for bigger values."""
    parts = []
    s = speed
    while s > 2.0:
        parts.append("atempo=2.0"); s /= 2.0
    while s < 0.5:
        parts.append("atempo=0.5"); s /= 0.5
    parts.append(f"atempo={s:.4f}")
    return ",".join(parts)


def silence_wav(path, seconds):
    run(["ffmpeg", "-y", "-f", "lavfi", "-i", "anullsrc=r=44100:cl=mono",
         "-t", f"{seconds:.3f}", "-c:a", "pcm_s16le", path])


def fit_clip(src_mp3, window, out_wav):
    """Speed-fit / pad a TTS clip to exactly `window` seconds of 44.1k mono wav."""
    dur = probe_duration(src_mp3)
    filters = []
    if dur > window and window > 0:
        filters.append(atempo_chain(min(dur / window, 4.0)))
    filt = (",".join(filters) + ",") if filters else ""
    filt += f"aresample=44100,apad,atrim=0:{window:.3f},asetpts=N/SR/TB"
    run(["ffmpeg", "-y", "-i", src_mp3, "-af", filt, "-c:a", "pcm_s16le", out_wav])


# ---------------------------------------------------------------- main entry
def run_job(job_dir, params, status_cb):
    """Run the whole pipeline for one job. status_cb(stage, percent, message)."""
    def log(msg):
        print(f"[job] {msg}", flush=True)

    video_path = params["video_path"]
    source_lang = norm_lang(params.get("source_lang") or "auto")
    target_lang = norm_lang(params.get("target_lang") or "my")
    gender = "male" if params.get("voice") == "male" else "female"
    aspect = params.get("aspect") or "original"
    keep_original = bool(params.get("keep_original_audio"))

    work = os.path.join(job_dir, "work")
    os.makedirs(work, exist_ok=True)

    # ---- 1. extract audio
    status_cb("extract", 5, "Getting the sound out of your video…")
    if not probe_has_audio(video_path):
        raise RuntimeError("This video has no sound track, so there is nothing to dub.")
    total_dur = probe_duration(video_path)
    audio_wav = os.path.join(work, "audio.wav")
    run(["ffmpeg", "-y", "-i", video_path, "-vn", "-ac", "1", "-ar", "16000",
         "-c:a", "pcm_s16le", audio_wav])

    # ---- 2. transcribe
    status_cb("transcribe", 15, f"Listening and writing down the speech (Whisper {WHISPER_MODEL})…")
    segments, detected = transcribe(audio_wav, source_lang)
    if not segments:
        raise RuntimeError("No speech was found in this video.")
    src_eff = norm_lang(detected)
    log(f"{len(segments)} segments, detected language: {detected}")
    status_cb("transcribe", 40, f"Found {len(segments)} lines of speech ({detected}).")

    # ---- 3. translate
    status_cb("translate", 45, f"Translating to {target_lang}…")
    translate_segments(segments, src_eff, target_lang, log)
    status_cb("translate", 58, "Translation done.")

    # ---- subtitles file (translated lines, original timing)
    srt_path = os.path.join(job_dir, "subtitles.srt")
    write_srt(segments, srt_path)

    # ---- 4+5. TTS each segment, fit to its window, assemble the dub track
    voice = pick_voice(target_lang, gender)
    log(f"TTS voice: {voice} (fallback: gTTS where supported)")
    tts_log_path = os.path.join(job_dir, "tts.log")
    tts_counts = {"edge-tts": 0, "gtts": 0, "silence": 0}

    def tlog(msg):
        log(msg)
        try:
            with open(tts_log_path, "a", encoding="utf-8") as fh:
                fh.write(msg + "\n")
        except Exception:
            pass

    parts, list_file = [], os.path.join(work, "concat.txt")
    cursor = 0.0
    n = len(segments)
    for i, seg in enumerate(segments):
        status_cb("tts", 60 + int(20 * (i + 1) / n),
                  f"Creating the new voice… line {i + 1} of {n}")
        start, end = seg["start"], max(seg["end"], seg["start"] + 0.4)
        window = end - start
        if start - cursor > 0.05:  # gap before this line -> silence
            gap = os.path.join(work, f"gap_{i}.wav")
            silence_wav(gap, start - cursor)
            parts.append(gap)
        seg_wav = os.path.join(work, f"seg_{i}.wav")
        mp3 = os.path.join(work, f"seg_{i}.mp3")
        try:
            if not seg["dub_text"].strip():
                raise ValueError("empty line")
            provider = tts_segment_fallback(seg["dub_text"], voice, target_lang,
                                            mp3, tlog)
            if provider is None:
                raise RuntimeError("all TTS providers failed")
            tts_counts[provider] += 1
            seg["tts_provider"] = provider
            tlog(f"line {i + 1}/{n}: voiced by {provider}")
            fit_clip(mp3, window, seg_wav)
        except Exception as e:  # one bad line must not kill the job
            tts_counts["silence"] += 1
            seg["tts_provider"] = "silence"
            tlog(f"line {i + 1}/{n}: SILENT ({e})")
            log(f"TTS failed for line {i + 1} ({e}); using silence")
            silence_wav(seg_wav, window)
        parts.append(seg_wav)
        cursor = end
    if total_dur - cursor > 0.05:
        tail = os.path.join(work, "tail.wav")
        silence_wav(tail, total_dur - cursor)
        parts.append(tail)
    with open(list_file, "w", encoding="utf-8") as f:
        for p in parts:
            f.write(f"file '{p}'\n")
    dub_wav = os.path.join(work, "dub.wav")
    run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", list_file,
         "-c:a", "pcm_s16le", dub_wav])

    # ---- 6. render final MP4
    status_cb("render", 85, "Putting picture, new voice and subtitles together…")
    out_path = os.path.join(job_dir, "output.mp4")
    vf = []
    if aspect == "9:16":
        vf.append("scale=720:1280:force_original_aspect_ratio=increase,crop=720:1280")
    elif aspect == "16:9":
        vf.append("scale=1280:720:force_original_aspect_ratio=increase,crop=1280:720")
    style = ("FontName=Noto Sans Myanmar,FontSize=18,PrimaryColour=&H00FFFFFF,"
             "OutlineColour=&H90000000,BorderStyle=1,Outline=2,Shadow=0,"
             "Alignment=2,MarginV=36")
    vf.append(f"subtitles={srt_path}:force_style='{style}'")
    cmd = ["ffmpeg", "-y", "-i", video_path, "-i", dub_wav]
    if keep_original:
        cmd += ["-filter_complex",
                f"[0:v]{','.join(vf)}[v];[0:a]volume=0.12[orig];"
                f"[1:a][orig]amix=inputs=2:duration=first:normalize=0[a]",
                "-map", "[v]", "-map", "[a]"]
    else:
        cmd += ["-vf", ",".join(vf), "-map", "0:v", "-map", "1:a"]
    cmd += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "21",
            "-c:a", "aac", "-b:a", "160k", "-shortest",
            "-movflags", "+faststart", out_path]
    run(cmd)

    tts_summary = (f"edge-tts {tts_counts['edge-tts']}/{n}, "
                   f"gTTS {tts_counts['gtts']}/{n}, "
                   f"silent {tts_counts['silence']}/{n}")
    tlog(f"TTS summary: {tts_summary}")
    log(f"TTS summary: {tts_summary}")
    status_cb("done", 100, "Done! Your dubbed video is ready to download.")
    return {"output": out_path, "srt": srt_path, "segments": len(segments),
            "detected_lang": detected, "voice": voice,
            "tts": tts_summary, "tts_counts": tts_counts}


In [ ]:
%%writefile server.py
"""Recap Studio v3 AUTO — server backend.

Upload a video -> the server transcribes it, translates it, dubs it with an
AI voice, burns subtitles and renders a finished MP4 you can download.
Pipeline lives in pipeline.py. Jobs run one at a time in a background thread
(CPU-only friendly); poll /api/status/{job_id} for progress.
"""
import os
import re
import shutil
import threading
import traceback
import uuid

# Sanitize proxy env vars (some HTTP libs crash on bracketed IPv6 entries)
for var in ("no_proxy", "NO_PROXY"):
    if os.environ.get(var):
        cleaned = [e.strip() for e in os.environ[var].split(",") if "[" not in e and "]" not in e]
        os.environ[var] = ",".join(cleaned)

from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.responses import FileResponse, HTMLResponse

import pipeline

HERE = os.path.dirname(os.path.abspath(__file__))
JOBS_DIR = os.path.join(HERE, "jobs")
os.makedirs(JOBS_DIR, exist_ok=True)

MAX_UPLOAD_MB = int(os.environ.get("MAX_UPLOAD_MB", "200"))
ALLOWED_EXT = {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"}

app = FastAPI(title="Recap Studio v3 AUTO")
jobs = {}            # job_id -> status dict (also persisted to status.json)
_jobs_lock = threading.Lock()
_run_lock = threading.Lock()   # one pipeline job at a time (CPU)


def _save_status(job_id):
    import json
    path = os.path.join(JOBS_DIR, job_id, "status.json")
    try:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(jobs[job_id], f, ensure_ascii=False)
    except Exception:
        pass


def _set_status(job_id, **kw):
    with _jobs_lock:
        jobs[job_id].update(kw)
        _save_status(job_id)


def _run_job(job_id, params):
    def cb(stage, percent, message):
        _set_status(job_id, stage=stage, percent=percent, message=message)
    with _run_lock:
        _set_status(job_id, stage="starting", percent=1, message="Starting…")
        try:
            result = pipeline.run_job(os.path.join(JOBS_DIR, job_id), params, cb)
            _set_status(job_id, stage="done", percent=100,
                        message="Done! Your dubbed video is ready to download.",
                        segments=result["segments"],
                        detected_lang=result["detected_lang"], voice=result["voice"],
                        tts=result.get("tts"), tts_counts=result.get("tts_counts"))
        except Exception as e:
            traceback.print_exc()
            _set_status(job_id, stage="error", percent=0,
                        message="Something went wrong while processing this video.",
                        error=str(e)[:500])


@app.get("/", response_class=HTMLResponse)
def index():
    with open(os.path.join(HERE, "static", "index.html"), encoding="utf-8") as f:
        return f.read()


@app.get("/api/health")
def health():
    return {"ok": True, "app": "recap-studio-v3", "whisper_model": pipeline.WHISPER_MODEL,
            "max_upload_mb": MAX_UPLOAD_MB,
            "languages": sorted(pipeline.LANGS.keys()),
            "edge_only_tts": sorted(c for c, e in pipeline.LANGS.items() if not e["gtts"])}


@app.post("/api/process")
async def process(file: UploadFile = File(...),
                  source_lang: str = Form("auto"),
                  target_lang: str = Form("my"),
                  voice: str = Form("female"),
                  aspect: str = Form("original"),
                  keep_original_audio: str = Form("false")):
    ext = os.path.splitext(file.filename or "")[1].lower()
    if ext not in ALLOWED_EXT:
        raise HTTPException(status_code=400,
                            detail=f"Please upload a video file ({', '.join(sorted(ALLOWED_EXT))}).")
    job_id = uuid.uuid4().hex[:12]
    job_dir = os.path.join(JOBS_DIR, job_id)
    os.makedirs(job_dir, exist_ok=True)
    # safe stored name: never trust the uploaded filename as a path
    safe_name = re.sub(r"[^A-Za-z0-9_.-]", "_", os.path.basename(file.filename))[:80] or "video"
    video_path = os.path.join(job_dir, "input" + ext)
    size = 0
    limit = MAX_UPLOAD_MB * 1024 * 1024
    with open(video_path, "wb") as out:
        while True:
            chunk = await file.read(1024 * 1024)
            if not chunk:
                break
            size += len(chunk)
            if size > limit:
                out.close()
                shutil.rmtree(job_dir, ignore_errors=True)
                raise HTTPException(status_code=400,
                                    detail=f"File too big (max {MAX_UPLOAD_MB} MB). "
                                           "Short clips work best.")
            out.write(chunk)
    with _jobs_lock:
        jobs[job_id] = {"job_id": job_id, "stage": "queued", "percent": 0,
                        "message": "Uploaded. Waiting to start…", "source_name": safe_name}
        _save_status(job_id)
    params = {"video_path": video_path, "source_lang": source_lang,
              "target_lang": target_lang, "voice": voice, "aspect": aspect,
              "keep_original_audio": str(keep_original_audio).lower() in ("true", "1", "yes")}
    threading.Thread(target=_run_job, args=(job_id, params), daemon=True).start()
    return {"job_id": job_id}


@app.get("/api/status/{job_id}")
def status(job_id: str):
    with _jobs_lock:
        st = jobs.get(job_id)
    if st is None:  # maybe the server restarted; fall back to disk
        import json
        path = os.path.join(JOBS_DIR, job_id, "status.json")
        if os.path.exists(path):
            with open(path, encoding="utf-8") as f:
                st = json.load(f)
        else:
            raise HTTPException(status_code=404, detail="Job not found.")
    return st


@app.get("/api/download/{job_id}")
def download(job_id: str):
    out = os.path.join(JOBS_DIR, job_id, "output.mp4")
    if not os.path.exists(out):
        raise HTTPException(status_code=404, detail="The video is not ready yet.")
    return FileResponse(out, media_type="video/mp4", filename="recap-dubbed.mp4")


@app.get("/api/subtitles/{job_id}")
def subtitles(job_id: str):
    srt = os.path.join(JOBS_DIR, job_id, "subtitles.srt")
    if not os.path.exists(srt):
        raise HTTPException(status_code=404, detail="Subtitles are not ready yet.")
    return FileResponse(srt, media_type="text/plain", filename="subtitles.srt")


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=int(os.environ.get("PORT", 7860)))


In [ ]:
!mkdir -p static jobs

In [ ]:
%%writefile static/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Recap Studio v3 AUTO — Upload · Generate · Download</title>
<style>
  :root{
    --bg:#07070c; --card:#12121e; --card2:#171728; --border:#2b2b44;
    --violet:#7c4dff; --violet2:#9d7bff; --gold:#ffc93c; --gold2:#e0a800;
    --text:#f2f1fa; --muted:#a5a3c0; --ok:#39d98a; --err:#ff6b7a;
  }
  *{box-sizing:border-box;margin:0;padding:0}
  body{background:radial-gradient(1000px 500px at 50% -80px,rgba(124,77,255,.22),transparent 60%),var(--bg);
       color:var(--text);font-family:system-ui,-apple-system,"Segoe UI",Roboto,"Noto Sans Myanmar",sans-serif;
       min-height:100vh}
  .wrap{max-width:680px;margin:0 auto;padding:16px 14px 44px}
  header{text-align:center;padding:20px 6px 4px}
  header h1{font-size:1.55rem;letter-spacing:.3px}
  header h1 .gold{color:var(--gold)}
  header p{color:var(--muted);font-size:.92rem;margin-top:6px}
  .pills{display:flex;gap:8px;justify-content:center;flex-wrap:wrap;margin-top:12px}
  .pill{border:1px solid var(--border);background:var(--card2);color:var(--muted);
        border-radius:999px;padding:5px 12px;font-size:.75rem}
  .pill.hot{border-color:var(--gold2);color:var(--gold)}
  .steps{display:flex;gap:6px;margin:16px 0 4px;flex-wrap:wrap}
  .step{flex:1;min-width:70px;text-align:center;border:1px solid var(--border);
        background:var(--card);border-radius:999px;padding:7px 4px;font-size:.72rem;color:var(--muted)}
  .step b{color:var(--gold);display:block;font-size:.85rem}
  .step.on{border-color:var(--violet);color:var(--text);box-shadow:0 0 14px rgba(124,77,255,.35)}
  .card{background:var(--card);border:1px solid var(--border);border-radius:18px;
        padding:18px;margin-top:14px;box-shadow:0 0 24px rgba(124,77,255,.10)}
  .card h2{font-size:1.02rem;margin-bottom:4px}
  .card h2 .n{color:var(--gold);margin-right:6px}
  .card .sub{color:var(--muted);font-size:.84rem;margin-bottom:10px}
  .drop{border:2px dashed var(--border);border-radius:14px;padding:26px 12px;text-align:center;
        cursor:pointer;background:var(--card2)}
  .drop:hover{border-color:var(--violet)}
  .drop .big{font-size:1.05rem;font-weight:600}
  .drop .small{color:var(--muted);font-size:.8rem;margin-top:5px}
  #fileName{color:var(--gold);font-size:.88rem;margin-top:10px;word-break:break-all}
  video{width:100%;border-radius:12px;margin-top:12px;display:none;background:#000}
  label{display:block;font-size:.84rem;color:var(--muted);margin:13px 0 6px}
  select{width:100%;background:#0a0a14;color:var(--text);border:1px solid var(--border);
         border-radius:12px;padding:13px;font-size:1rem}
  .row{display:flex;gap:10px}.row>div{flex:1}
  .check{display:flex;align-items:center;gap:10px;margin-top:14px;font-size:.9rem;color:var(--text)}
  .check input{width:20px;height:20px;accent-color:var(--violet)}
  .gen{width:100%;margin-top:18px;background:linear-gradient(135deg,var(--violet),#5b2ee5);
       color:#fff;border:none;border-radius:999px;padding:16px;font-size:1.08rem;font-weight:700;
       cursor:pointer;box-shadow:0 6px 22px rgba(124,77,255,.4)}
  .gen:disabled{opacity:.45;cursor:not-allowed;box-shadow:none}
  .barwrap{background:#0a0a14;border:1px solid var(--border);border-radius:999px;height:20px;
           overflow:hidden;margin-top:14px;display:none}
  .bar{height:100%;width:0%;background:linear-gradient(90deg,var(--violet),var(--gold));
       transition:width .5s ease}
  #statusMsg{font-size:.9rem;margin-top:10px;color:var(--text);display:none}
  #statusMsg.err{color:var(--err)}
  .dl{display:none;width:100%;margin-top:14px;background:linear-gradient(135deg,#1fae62,#128a4c);
      color:#fff;text-align:center;text-decoration:none;border-radius:999px;padding:16px;
      font-size:1.05rem;font-weight:700}
  .srt{display:none;text-align:center;margin-top:10px}
  .srt a{color:var(--gold);font-size:.88rem}
  .note{background:rgba(124,77,255,.08);border:1px solid rgba(124,77,255,.35);border-radius:12px;
        padding:11px 14px;font-size:.83rem;color:#cfc4ff;margin-top:14px;line-height:1.55}
  footer{color:var(--muted);font-size:.75rem;text-align:center;margin-top:18px}
</style>
</head>
<body>
<div class="wrap">
  <header>
    <h1>🎬 Recap Studio <span class="gold">v3 AUTO</span></h1>
    <p>Upload a clip → Generate → Download. အလိုအလျောက် ဘာသာပြန် + အသံသစ် + စာတန်း</p>
    <div class="pills">
      <span class="pill hot">AUTO dub</span><span class="pill">Auto translate</span>
      <span class="pill">Auto subtitles</span><span class="pill">Server render</span>
    </div>
  </header>

  <div class="steps">
    <div class="step on" id="st1"><b>1</b>Upload</div>
    <div class="step" id="st2"><b>2</b>Languages &amp; voice</div>
    <div class="step" id="st3"><b>3</b>Generate</div>
    <div class="step" id="st4"><b>4</b>Download</div>
  </div>

  <div class="card">
    <h2><span class="n">01</span>Upload your video · ဗီဒီယိုတင်ပါ</h2>
    <div class="sub">MP4, MOV, MKV, WebM — up to 200 MB. Short clips (under ~3 minutes) work best.</div>
    <div class="drop" id="drop">
      <div class="big">📁 Tap to choose a video</div>
      <div class="small">The server will listen, translate, dub and edit it for you.</div>
      <div id="fileName"></div>
    </div>
    <input type="file" id="file" accept="video/*,.mkv,.avi" style="display:none">
    <video id="preview" controls playsinline></video>
  </div>

  <div class="card">
    <h2><span class="n">02</span>Languages &amp; voice · ဘာသာစကားနှင့် အသံ</h2>
    <label>Video language (what people speak in the clip)</label>
    <select id="src">
      <option value="auto" selected>Auto-detect</option>
      <option value="en">English · en</option>
      <option value="my">Burmese · မြန်မာ · my</option>
      <option value="th">Thai · ไทย · th</option>
      <option value="zh-CN">Chinese · 中文 · zh</option>
      <option value="ja">Japanese · 日本語 · ja</option>
      <option value="ko">Korean · 한국어 · ko</option>
      <option value="hi">Hindi · हिन्दी · hi</option>
      <option value="id">Indonesian · Bahasa Indonesia · id</option>
      <option value="vi">Vietnamese · Tiếng Việt · vi</option>
      <option value="ms">Malay · Bahasa Melayu · ms</option>
      <option value="es">Spanish · Español · es</option>
      <option value="fr">French · Français · fr</option>
      <option value="de">German · Deutsch · de</option>
      <option value="ar">Arabic · العربية · ar</option>
      <option value="bn">Bengali · বাংলা · bn</option>
      <option value="ta">Tamil · தமிழ் · ta</option>
      <option value="te">Telugu · తెలుగు · te</option>
      <option value="ur">Urdu · اردو · ur</option>
      <option value="pt">Portuguese · Português · pt</option>
      <option value="ru">Russian · Русский · ru</option>
      <option value="it">Italian · Italiano · it</option>
      <option value="fil">Filipino · fil</option>
      <option value="km">Khmer · ខ្មែរ · km</option>
      <option value="lo">Lao · ລາວ · lo</option>
    </select>
    <div class="row">
      <div>
        <label>Dub language (new voice speaks…)</label>
        <select id="tgt">
          <option value="my" selected>Burmese · မြန်မာ · my</option>
          <option value="en">English · en</option>
          <option value="th">Thai · ไทย · th</option>
          <option value="zh-CN">Chinese · 中文 · zh</option>
          <option value="ja">Japanese · 日本語 · ja</option>
          <option value="ko">Korean · 한국어 · ko</option>
          <option value="hi">Hindi · हिन्दी · hi</option>
          <option value="id">Indonesian · Bahasa Indonesia · id</option>
          <option value="vi">Vietnamese · Tiếng Việt · vi</option>
          <option value="ms">Malay · Bahasa Melayu · ms</option>
          <option value="es">Spanish · Español · es</option>
          <option value="fr">French · Français · fr</option>
          <option value="de">German · Deutsch · de</option>
          <option value="ar">Arabic · العربية · ar</option>
          <option value="bn">Bengali · বাংলা · bn</option>
          <option value="ta">Tamil · தமிழ் · ta</option>
          <option value="te">Telugu · తెలుగు · te</option>
          <option value="ur">Urdu · اردو · ur</option>
          <option value="pt">Portuguese · Português · pt</option>
          <option value="ru">Russian · Русский · ru</option>
          <option value="it">Italian · Italiano · it</option>
          <option value="fil">Filipino · fil</option>
          <option value="km">Khmer · ខ្មែរ · km</option>
          <option value="lo">Lao · ລາວ · lo</option>
        </select>
      </div>
      <div>
        <label>Voice · အသံ</label>
        <select id="voice">
          <option value="female" selected>Female · အမျိုးသမီး</option>
          <option value="male">Male · အမျိုးသား</option>
        </select>
      </div>
    </div>
    <label>Video shape</label>
    <select id="aspect">
      <option value="original" selected>Original size</option>
      <option value="9:16">9:16 — Reels / TikTok</option>
      <option value="16:9">16:9 — YouTube</option>
    </select>
    <label class="check"><input type="checkbox" id="keepOrig">
      Keep original sound quietly underneath the dub</label>
    <button class="gen" id="generate">✨ Generate my dubbed video</button>
    <div class="barwrap" id="barwrap"><div class="bar" id="bar"></div></div>
    <div id="statusMsg"></div>
    <a class="dl" id="download" href="#">⬇️ Download dubbed video</a>
    <div class="srt" id="srtBox"><a id="srtLink" href="#">Also download the subtitles (.srt)</a></div>
  </div>

  <div class="note">
    ℹ️ Honest notes: processing happens on the server (not on your phone), so stay on
    this page while it works — a 1-minute clip can take a few minutes on a free server.
    The AI voice is a synthetic voice, not a real person. Translation is automatic —
    check the wording before you post. Short clips work best.
  </div>

  <footer>Recap Studio v3 AUTO · made for Zomi creators</footer>
</div>

<script>
const $ = id => document.getElementById(id);
let fileObj = null, timer = null;

$('drop').addEventListener('click', () => $('file').click());
$('file').addEventListener('change', () => {
  fileObj = $('file').files[0] || null;
  if (!fileObj) return;
  $('fileName').textContent = '🎞️ ' + fileObj.name;
  const v = $('preview');
  v.src = URL.createObjectURL(fileObj);
  v.style.display = 'block';
  markStep(2);
});

function markStep(n){
  for (let i = 1; i <= 4; i++) $('st' + i).classList.toggle('on', i <= n);
}
function setStatus(txt, isErr){
  const el = $('statusMsg');
  el.style.display = 'block';
  el.textContent = txt;
  el.classList.toggle('err', !!isErr);
}

$('generate').addEventListener('click', async () => {
  if (!fileObj) { setStatus('Choose a video first. / ဗီဒီယို အရင်ရွေးပါ', true); return; }
  $('generate').disabled = true;
  $('download').style.display = 'none';
  $('srtBox').style.display = 'none';
  $('barwrap').style.display = 'block';
  $('bar').style.width = '2%';
  markStep(3);
  setStatus('Uploading…');
  const fd = new FormData();
  fd.append('file', fileObj);
  fd.append('source_lang', $('src').value);
  fd.append('target_lang', $('tgt').value);
  fd.append('voice', $('voice').value);
  fd.append('aspect', $('aspect').value);
  fd.append('keep_original_audio', $('keepOrig').checked ? 'true' : 'false');
  let job;
  try {
    const r = await fetch('/api/process', { method: 'POST', body: fd });
    const data = await r.json();
    if (!r.ok) throw new Error(data.detail || 'Upload failed');
    job = data.job_id;
  } catch (e) {
    setStatus('❌ ' + e.message, true);
    $('generate').disabled = false;
    return;
  }
  clearInterval(timer);
  timer = setInterval(async () => {
    try {
      const r = await fetch('/api/status/' + job);
      const s = await r.json();
      $('bar').style.width = Math.max(2, s.percent || 0) + '%';
      if (s.stage === 'done') {
        clearInterval(timer);
        markStep(4);
        setStatus('✅ Done! Your dubbed video is ready.');
        const d = $('download');
        d.href = '/api/download/' + job;
        d.style.display = 'block';
        $('srtLink').href = '/api/subtitles/' + job;
        $('srtBox').style.display = 'block';
        $('generate').disabled = false;
      } else if (s.stage === 'error') {
        clearInterval(timer);
        setStatus('❌ ' + (s.error || s.message || 'Processing failed'), true);
        $('generate').disabled = false;
      } else {
        setStatus('⏳ ' + (s.message || 'Working…') + ' (' + (s.percent || 0) + '%)');
      }
    } catch (e) { /* keep polling through brief network drops */ }
  }, 2500);
});
</script>
</body>
</html>


In [ ]:
# Step 2 — download cloudflared (free public tunnel, no account needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared && echo OK

In [ ]:
# Step 3 — start the website and publish it on a public link
import subprocess, threading, time, re, urllib.request, os
os.environ.setdefault('WHISPER_MODEL', 'small')  # good balance on free CPU Colab

def _run():
    import uvicorn
    from server import app
    uvicorn.run(app, host='127.0.0.1', port=7860, log_level='warning')

threading.Thread(target=_run, daemon=True).start()
print('Starting server...', flush=True)

for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:7860/api/health', timeout=5)
        print('Server is up.')
        break
    except Exception:
        time.sleep(5)
else:
    raise SystemExit('Server did not start — re-run the cells above.')

proc = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

url = None
for line in proc.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m and 'api.' not in m.group(0):
        url = m.group(0)
        break

if url:
    print('\n' + '=' * 52)
    print('YOUR WEBSITE IS LIVE AT:')
    print(url)
    print('=' * 52)
    print('\nOpen it on your phone. Keep this Colab tab open.')
else:
    raise SystemExit('Tunnel failed — re-run this cell.')

# keep the tunnel alive while this cell runs
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    pass
